<a href="https://colab.research.google.com/github/MrFire24/Dataset-Organizer/blob/main/dataset_organizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# @title 1. Загрузка датасета {"single-column":true, run:"auto"}

source_type = "Google Drive Link" #@param ["Upload File", "Google Drive", "Google Drive Link"]
#add_job_json == False #@param {type:"boolean"}
# @markdown Выбирав опцию "Google Drive" будет запрошен доспут к вашему диску

# @markdown Также в этом случае файл должен находится на вашем Google диске

from IPython.core.interactiveshell import dis
import ipywidgets as widgets
from IPython.display import display
import os

# Файл который мы пытаемся получить для дальнейшего парсинга
data_file_path = ""

##########################

def get_file_local():
  from google.colab import files
  print("\nВнимание!")
  print("Загрузка больших файлов таким методом может занять много времени")
  print("Повторная загрузка файла создаёт дубликат\n")

  upl_files = files.upload('uploaded_data')
  if len(upl_files.keys()) > 1:
    print("Слишком много файлов!")
  else:
    print(upl_files.keys())
    full_path = "/content/" + list(upl_files.keys())[0]
    if os.path.exists(full_path) and full_path.lower().endswith('.csv'):
      global data_file_path
      data_file_path = full_path
      print("Файл успешно загружен!")
    else:
      print("Файл не найден! Что-то пошло не так")
      print(full_path)

##########################

def extract_file_id(value):
    if '/d/' in value:
        return value.split('/d/')[1].split('/')[0]
    return value

file_link_input = widgets.Text(placeholder="Введите ссылку")
confirm_button_link = widgets.Button(description="Подтвердить", icon='check')
def on_confirm_button_link(button):
  import gdown
  file_link = extract_file_id(file_link_input.value)
  url = f'https://drive.google.com/uc?id={file_link}'
  output = "uploaded_data/"

  full_path = ""
  try:
    full_path = "/content/" + gdown.download(url, output, quiet=False)
  except:
    print("Файл недоступен или указано неверное ссылка")
    button.button_style = "danger"
  else:
    print()
    if os.path.exists(full_path):
      if full_path.lower().endswith('.csv'):
        print("Файл успешно загружен!")
        button.button_style = "success"
        global data_file_path
        data_file_path = full_path
        print("Вы можете запустить следующий блок кода для начала парсинга")
      else:
        print("Файл найден, но это НЕ CSV файл")
        button.button_style = "warning"
    else:
      print("Файл не найден! Что-то пошло не так")
      print(full_path)
      button.button_style = "danger"

def get_file_from_drive_link():
  # Создание виджета ввода
  print("Введите ссылку на файл с Google Drive")
  print("Из ссылки будет получено ID файла для скачивания")
  print("Убедитесь что в ссылке есть \"d/\"")
  confirm_button_link.on_click(on_confirm_button_link)
  display(file_link_input, confirm_button_link)

########################

file_path_input = widgets.Text(placeholder="Введите путь к файлу")
confirm_button_path = widgets.Button(description="Подтвердить", icon='check')
def on_confirm_button_path(button):
  full_path = "/content/drive/MyDrive/" + file_path_input.value
  if os.path.exists(full_path):
    if full_path.lower().endswith('.csv'):
      print("Файл найден")
      button.button_style = "success"
      global data_file_path
      data_file_path = full_path
      print("Вы можете запустить следующий блок кода для начала парсинга")
    else:
      print("Файл найден, но это НЕ CSV файл")
      button.button_style = "warning"
  else:
    print("Файл не найден по указанному пути")
    print("/content/drive/" + file_path_input.value)
    button.button_style = "danger"

def get_file_from_drive():
  # Подключение к диску
  from google.colab import drive
  drive.mount("/content/drive", force_remount=True)

  # Создание виджета ввода
  print("Введите путь к вашему файлу (filename.csv)")
  print("Если он находиться в какой-то папке на диске то путь должен вышлядить так:")
  print("\"folder/filename.csv\"")

  confirm_button_path.on_click(on_confirm_button_path)
  display(file_path_input, confirm_button_path)


################

if source_type == "Upload File":
  get_file_local()
elif source_type == "Google Drive":
  get_file_from_drive()
elif source_type == "Google Drive Link":
  get_file_from_drive_link()


In [ ]:
# @title 2. Парсинг {"single-column":true}

#jbi_name
CUSTOM_DATA_PATH = "" #@param {type:"string", placeholder:"Оставьте пустым если использовали блок выше"}
CUSTOM_NAME = "" #@param {type:"string", placeholder:"По стандарту берёт первое значение jbi_name"}
# @markdown ---
OUTPUT_PATH = "drive/MyDrive/processed_data " #@param {type:"string"}
STRIP_PREFIXES = True #@param {type:"boolean"}

from google.colab import files
import pandas as pd
from pathlib import Path

# Гарантируем наличие папки для вывода
os.makedirs(OUTPUT_PATH, exist_ok=True)

def strip_prefixes(df, group):
    return df.rename(columns=lambda col: col.removeprefix(group + '_') if col != SYS_TIME else col)

def get_jbi_name(df):
    if 'control_jbi_name' in df.columns:
        name = df['control_jbi_name'].iloc[0]
        if not pd.isna(name) and str(name).strip() != '':
            return str(name).strip()
    return 'UNNAMED'

SYS_TIME = 'sys_time'
GROUPS = ['coordinate', 'io', 'control', 'weld', 'scanner', 'termo', 'set', 'positioner']

def process_csv(file):
  df = pd.read_csv(file)

  jbi_name = CUSTOM_NAME.strip() if CUSTOM_NAME.strip() else get_jbi_name(df)
  start_time = pd.to_datetime(df[SYS_TIME].iloc[0]).strftime('%Y-%m-%d_%H-%M-%S')

  session_path = Path(OUTPUT_PATH) / jbi_name / start_time

  if session_path.exists():
    answer = input(f"Папка {session_path} уже существует. Перезаписать? (y/n): ")
    if answer.lower() != 'y':
        print("Парсинг отменён.")
        return False

  session_path.mkdir(parents=True, exist_ok=True)

  for group in GROUPS:
      cols = [SYS_TIME] + [col for col in df.columns if col.startswith(group + '_')]
      if len(cols) > 1:
          result = df[cols]
          if STRIP_PREFIXES:
              result = strip_prefixes(result, group)
          result.to_csv(session_path / f'{group}.csv', index=False)
  return True

def process_json(file):
  pass

# --- Мэйн ---
if 'data_file_path' not in dir() or not data_file_path:
    data_file_path = None

if CUSTOM_DATA_PATH:
    data_file_path = "/content/" + CUSTOM_DATA_PATH

if data_file_path and Path(data_file_path).exists():
    if process_csv(data_file_path):
        print("Парсинг завершен!")
else:
    print("Ошибка: Путь к файлу пуст или файл не найден.")
    if not CUSTOM_DATA_PATH:
        print("Сначала загрузите файл в блоке выше.")
    else:
        print("Проверьте указанный вами путь.")
    print(data_file_path)

In [ ]:
# @title 3. Скачивание результата {"single-column":true}

download_type = "Google Drive" #@param ["Архив", "Архив в Colab", "Google Drive"]
DRIVE_PATH = "Dataset Organizer Results" #@param {type:"string", placeholder:"Путь на Google Drive"}
#@markdown Название папки куда сохраняет результат при выборе варианта "Google Drive"

import shutil
from pathlib import Path
from google.colab import files

if download_type == "Архив":
    shutil.make_archive('results', 'zip', OUTPUT_PATH)
    files.download('results.zip')

elif download_type == "Google Drive":
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

    drive_path = Path('/content/drive/MyDrive') / DRIVE_PATH

    if drive_path.exists():
        answer = input(f"Папка {drive_path} уже существует. Перезаписать? (y/n): ")
        if answer.lower() != 'y':
            print("Скачивание отменено.")
        else:
            shutil.rmtree(drive_path)
            shutil.copytree(OUTPUT_PATH, drive_path)
            print(f"Результаты сохранены в {drive_path}")
    else:
        shutil.copytree(OUTPUT_PATH, drive_path)
        print(f"Результаты сохранены в {drive_path}")

elif download_type == "Архив в Colab":
    shutil.make_archive('results', 'zip', OUTPUT_PATH)
    print("Архив сохранён в /content/results.zip")

# Далее находится служебные блоки кода и наброски

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
df = pd.read_csv(data_file_path)
display(df)

In [ ]:
from logging import warn
from google.colab import files
import pandas as pd
from pathlib import Path

# --- Настройки ---

OUTPUT_PATH = Path('processed_data')
STRIP_PREFIXES = True
IMPORT_FROM_PATH = True
IMPORT_PATH = Path('uploaded_data/dataset_all_fields.csv')

# --- Функции ---

def strip_prefixes(df, group):
    return df.rename(columns=lambda col: col.removeprefix(group + '_') if col != SYS_TIME else col)

SYS_TIME = 'sys_time'
GROUPS = ['coordinate', 'io', 'control', 'weld', 'scanner', 'termo', 'set', 'positioner']

def process_csv(file):
  df = pd.read_csv(file)
  for group in GROUPS:
      cols = [SYS_TIME] + [col for col in df.columns if col.startswith(group + '_')]
      if len(cols) > 1:
          result = df[cols]
          if STRIP_PREFIXES:
              result = strip_prefixes(result, group)
          result.to_csv(OUTPUT_PATH / f'{group}.csv', index=False)

def process_json(file):
  pass

# --- Мэйн ---
if (IMPORT_FROM_PATH):
  process_csv(IMPORT_PATH)
else:
  OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
  uploaded = files.upload('uploaded_data')

  for fn in uploaded.keys():
    print('User uploaded file "{name}" with length {length} bytes'.format(
        name=fn, length=len(uploaded[fn])))
  print()

  for file in uploaded:
    if file.lower().endswith('.csv'):
      process_csv(file)
    elif file.lower().endswith('.json'):
      process_json(file)
    else:
      warn(f'Unsupported file type: {file}')
